In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# One row per order (output of 06_shipment_cdc); tracking columns are NULL when the order has no event
status_history = spark.table(
    "logistics.silver.silver_shipment_status_history"
)

# Lifecycle analysis works on actual tracking events only
events_df = status_history.filter(col("tracking_event_id").isNotNull())

print("Orders in status history:", status_history.count())
print("Rows with a tracking event:", events_df.count())
display(events_df)

In [0]:
lifecycle_window = Window.partitionBy(
    "shipment_id"
).orderBy(
    col("event_ts").asc(),
    col("received_ts").asc_nulls_last(),
    col("tracking_event_id").asc()
)

lifecycle_df = events_df.withColumn(
    "lifecycle_sequence",
    row_number().over(lifecycle_window)
)

display(
    lifecycle_df.orderBy(
        "shipment_id",
        "lifecycle_sequence"
    )
)

In [0]:
lifecycle_df = lifecycle_df.withColumn(
    "previous_event",
    col("shipment_status")  # status of the shipment before CDC
)

lifecycle_df = lifecycle_df.withColumn(
    "transition_status",
    when(
        col("previous_event") == "DELIVERED",
        "INVALID"
    ).otherwise("VALID_OR_UNCHECKED")
)

display(lifecycle_df)

In [0]:
valid_lifecycle = lifecycle_df.filter(col("transition_status") != "INVALID")

lifecycle_exceptions = lifecycle_df.filter(col("transition_status") == "INVALID").withColumn(
    "exception_reason",
    lit("Shipment has another event after DELIVERED")
)

In [0]:
# Stage duration = time from the previous event (prev_event_ts) to the current event (event_ts)
lifecycle_df = lifecycle_df.withColumn(
    "stage_duration_hours",
    (
        unix_timestamp("event_ts") -
        unix_timestamp("prev_event_ts")
    ) / 3600
)

display(
    lifecycle_df.select(
        "shipment_id",
        "prev_event_type",
        "prev_event_ts",
        "event_type",
        "event_ts",
        "stage_duration_hours"
    )
)

In [0]:
# Simple per-shipment summary: one row per order (left join keeps orders without events)
event_summary = lifecycle_df.groupBy("shipment_id").agg(
    count("*").alias("total_events"),
    max("event_ts").alias("last_event_ts")
)

shipment_summary = (
    status_history.select(
        "shipment_id",
        "carrier_id",
        col("shipment_status").alias("previous_status"),
        col("updated_shipment_status").alias("current_status"),
        "cdc_operation"
    )
    .join(event_summary, on="shipment_id", how="left")
    .withColumn("total_events", coalesce(col("total_events"), lit(0)))
)

# Small overview: shipments per current status
display(
    shipment_summary.groupBy("current_status")
    .agg(
        count("*").alias("shipments"),
        sum("total_events").alias("tracking_events")
    )
    .orderBy(col("shipments").desc())
)

display(shipment_summary)

In [0]:
lifecycle_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_lifecycle"
    )

In [0]:
lifecycle_exceptions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_lifecycle_exceptions"
    )

In [0]:
shipment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_lifecycle_summary"
    )

for t in ["silver_shipment_lifecycle", "silver_lifecycle_exceptions", "silver_shipment_lifecycle_summary"]:
    print(t, spark.table(f"logistics.silver.{t}").count())